# Assignment 5: Autoencoders
Date: 

Report produced by:  
COMP 3705 Group 3

## Data Preprocessing

### Data Loading

In [ ]:
import pandas as pd
import numpy as np
import math as m
import tensorflow as tf
from sklearn.model_selection import train_test_split as tts
from keras.models import Model, Sequential
from keras.layers import Input, Dense
import matplotlib.pyplot as plt
# Set TensorFlow deprication warnings to False
import tensorflow.python.util.deprecation as deprecation
deprecation._PRINT_DEPRECATION_WARNINGS = False

# Load the Dataset and Check Shape
df = pd.read_csv("csvImages 10k x 784.csv")
df_clean = df.copy()
print("Data Shape:", df_clean.shape)

The data was obtained from Kaggle, and downloaded form [Learn](https://learn.rrc.ca/d2l/le/content/759307/Home). It was said to contain 10,000 images of size 28x28 (784 pixels), with each image being flattened as a 1D-array. The individual values were said to be recorded as grayscale so we expected there to be exactly 784 columns. Upon inspecting the shape of the data, we can confirm that there are 784 columns, coincidign with our assumptions of it being grayscale and will proceed accordingly. Of the 10,000 rows, we found that one was missing. As we still have enough data for training our encoder, this will not cause any problems going forward.

### Check Data Integrity

In [ ]:
print("Number of Duplicate Rows:",df_clean.duplicated().sum())
print("Min value across all columns:",df_clean.min().min())
print("Max value across all columns:",df_clean.max().max())
print("Number of missing values in data:",df_clean.isna().sum().sum())

As the data across all columns ranges between 0 and 255, we can determine that there are no unexpected RGB values.  
Additionally, since there are also no missing values anywhere in the data we can proceed with the dataset as-is.

In [ ]:
df_clean = df_clean.drop_duplicates()
print("Cleaned Data Shape:", df_clean.shape)

### Reshape for Visualizations

In [ ]:
# Convert DataFrame to Numpy Array and Normalize Values
img_data = df_clean.values/255
img_data = img_data.reshape((-1,28,28))
print("Shape of Array:",img_data.shape)
print("Shape of Each Row:",img_data[0].shape)

After checking the data, we converted the DataFrame into a NumPy array and normalized the pixel values by dividing them by 255. This changed the value range from 0–255 to  0–1. Normalization was done to helps the neural network train more effectively by ensuring no one pixel disproportionately influences the model.  
Next, we reshaped each image from 784 values into a 28 × 28 image. We did this because each row contains 784 points, each pertaining to a pixel in a 28x28 image. Reshaping the data allows us to display the images using Matplotlib before training the autoencoder.

### Pre-Encoding Visualizations

In [ ]:
plt.figure(figsize=(16,10))
for i in range(5):
    plt.subplot(1,5,i+1)
    plt.imshow(img_data[i],cmap="gray")
    plt.axis("off")

We plotted the first five images using using a grayscale colormap. These original images serve as the baseline we will compare the decoded images against when judging the autoencoder.  
Each image shows a light digit stroke on a dark background, roughly centred in the frame. Most pixels, especially around the edges, are 0 (black) and carry no information about the digit.  
Much of each image is "empty" space, which suggests the images can be compressed substantially without losing the digit itself.

### Reshape for Data Encoding

In [ ]:
img_data = img_data.reshape((-1,28**2))
print("Shape of Array:",img_data.shape)
print("Shape of Each Row:",img_data[0].shape)

We flattened each 28×28 image back into a one-dimensional array of 784 values. Keras `Dense` layers take one-dimensional input, so the autoencoder's input layer expects 784 values per image.  
The data shape is back to **(9998, 784)**, with values between 0 and 1, which is ready to feed into the network. 

In [ ]:
# Dummy Cell - Used for Run All Above - Remove Before Submission

## Model Selection

In [ ]:
def compile_autoencoder(data,autoencoder,weights,validation=None,optimizer="adadelta",loss="binary_crossentropy",epochs=50):
    if weights != None:
        autoencoder["decoded"].set_weights(weights)
    
    autoencoder["decoded"].compile(optimizer=optimizer,loss=loss)
    autoencoder["decoded"].fit(data,data,validation_data=validation,epochs=epochs)

def create_autoencoder(data,activation_function="relu",activation_final="sigmoid",loss_function="binary_crossentropy",optimizer="adadelta",weights = None, epochs=50):
    autoencoder = dict()
    train_data, test_data = tts(data,train_size=0.9,test_size=0.1,random_state=42)
    # Define Input Layer
    input_layer = Input(shape=(784,))
    # Define Encoding Layers
    hidden_encoding1 = Dense(1024,activation=act)(input_layer)
    hidden_encoding2 = Dense(512,activation=act)(hidden_encoding1)
    hidden_encoding3 = Dense(256,activation=act)(hidden_encoding2)
    hidden_encoding4 = Dense(128,activation=act)(hidden_encoding3)
    # Final Encoding Layer
    encoding_stage = Dense(64,activation=act)(hidden_encoding4)
    # Define Decoding Layers
    hidden_decoding1 = Dense(128,activation=act)(encoding_stage)
    hidden_decoding2 = Dense(256,activation=act)(hidden_decoding1)
    hidden_decoding3 = Dense(512,activation=act)(hidden_decoding2)
    hidden_decoding4 = Dense(1024,activation=act)(hidden_decoding3)
    # Final Decoding Layer
    decoding_stage = Dense(784,activation=act_fnl)(hidden_decoding4)
    # Define Autoencoder Model
    autoencoder = {"encoded": Model(input_layer,encoding_stage),
                    "decoded": Model(input_layer,decoding_stage),
                    "Final Act": act_fnl,
                    "Hidden Act": act,
                    "Loss Function": loss,
                    "Optimizer": opti,
    }
    # Set Starting Weights
    initial_weights = weights
        
    # Compile and Train Model
    compile_autoencoder(train_data,autoencoder,initial_weights,optimizer=opti,loss=loss,epochs=epochs)
    autoencoder["Loss"] = autoencoder["decoded"].evaluate(test_data,test_data,verbose=0)
    return autoencoder

def plot_images(data, autoencoders,num_img=5):
    data = data.reshape((-1,28**2))
    for key in autoencoders:
        encoded_img = autoencoders[key]["encoded"].predict(data[:num_img]).reshape((-1,8,8))
        decoded_img = autoencoders[key]["decoded"].predict(data[:num_img]).reshape((-1,28,28))
        loss = autoencoders[key]["Loss Function"]
        hidden_act = autoencoders[key]["Hidden Act"]
        final_act = autoencoders[key]["Final Act"]
        data = data.reshape((-1,28,28))
        # print("Autoencoder ID:",i)
        plt.figure(figsize=(16,10))
        for j in range(num_img):
            plt.subplot(3,num_img,j+1)
            plt.imshow(data[j],cmap="gray")
            plt.axis("off")

            plt.subplot(3,num_img,j+1+num_img)
            plt.imshow(encoded_img[j],cmap="gray")
            plt.axis("off")

            plt.subplot(3,num_img,j+1+num_img*2)
            plt.imshow(decoded_img[j],cmap="gray")
            plt.axis("off")
        data = data.reshape((-1,28**2))

In [ ]:
activation_function = ["relu","sigmoid"]
activation_final = ["sigmoid","softmax"]
loss_functions = ["binary_crossentropy","mean_squared_error","mean_absolute_error"]
optimizer = ["adadelta"]
autoencoders1 = dict()
i=0
for act in activation_function:
    for act_fnl in activation_final:
        for loss in loss_functions:
            for opti in optimizer:
                if i != 0:
                    w = autoencoders1[0]["decoded"].get_weights()
                else:
                    w = None
                autoencoders1[i] = create_autoencoder(img_data,act,act_fnl,loss,opti,w)
                i += 1
df_encoded_pars1 = pd.DataFrame(autoencoders1).transpose().drop(columns=["decoded","encoded"])

In [ ]:
plot_images(img_data,autoencoders1)
df_encoded_pars1

We tested 12 autoencoder models using different combinations of hidden-layer activation functions, final-layer activation functions, and loss functions.  
The hidden-layer activation functions were RELU and Sigmoid. The final-layer activation functions were Sigmoid and Softmax. We also tested Binary Crossentropy, Mean Squared Error (MSE), and Mean Absolute Error (MAE) as the different loss functions.

### Autoencoder 0: ReLU / Sigmoid / Binary Cross-Entropy
ReLU in the hidden layers, Sigmoid on the output layer, and Binary Cross-Entropy as the loss function.
The Training loss fell steadily from **0.2845** to **0.0775** over 50 epochs and in the plot, the decoded images kept the overall shape and position of each original digit, though strokes are softer and blurrier than in the originals. The encoded row shows 8×8 grids of light and dark squares rather than a digit. We accepted this model as the loss decreased consistently and the reconstructions are recognizable. Also ReLU helps reduce the vanishing gradient problem, allowing the deeper layers to learn more effectively. Sigmoid is used in the output layer because our image pixels were normalized between 0 and 1, and it keeps the reconstructed pixel values within that same range. This combination may have helped the autoencoder learn and reconstruct the images effectively.

### Autoencoder 1: ReLU / Sigmoid / Mean Squared Error
Same activation functions as Autoencoder 0 but with Mean Squared Error as the loss function. Training loss dropped from **0.0958** to **0.0121** over 50 epochs. It briefly levelled off around 0.055 in epochs 2–4, then declined steadily and was still improving at epoch 50. In the plot, the decoded digits closely follow the originals but compared to autoencoder 1 the loss of sharpness is greater along the edges. MSE penalizes large pixel errors more heavily, encouraging the autoencoder to reconstruct the original image more accurately and like Autoencoder 0, its loss was still decreasing at epoch 50, suggesting that the model was still learning and might improve with more training. 
has context menu

### Rejected Autoencoders
After comparing the original and reconstructed images, we found that Autoencoders 2, 4, 5, 6, 7, 8, 9, 10, and 11 did not perform well.  

**Model 2:** RELU, Sigmoid, and MAE  
- The reconstructed digits were less clear than the original images. Some of the lines appeared blurry, and the edges were not as sharp. Parts of the handwritten strokes were difficult to distinguish. We rejected this model because it did not preserve the original digit shapes as clearly as the other models.  

**Models 4 and 5:**
- These models used RELU in the hidden layers and Softmax in the final layer. Model 4 used MSE, while Model 5 used MAE. The reconstructed images looked like gray, grainy noise instead of handwritten digits. They contained random-looking light and dark pixels, with no clear lines, curves, or recognizable digit shapes. The outputs also looked almost identical even though the original images showed different digits. We rejected these models because they failed to reconstruct recognizable digits using the architecture and settings tested.  

**Models 6, 7, and 9**
- These models used Sigmoid for both the hidden layers and the final layer. Model 6 used Binary Crossentropy as the loss function while Model 7 used MSE. Model 9 also had sigmoid in the hidden layer but used softmax as the final layers activation function. like Model 6, it used Binary Crossentropy as the loss function. The reconstructed images produced by these models were very blurry, and some looked like bright, cloudy shapes instead of handwritten digits. The original lines and curves were difficult to see. Another problem was that several reconstructed images looked almost identical, even though the original digits were different. This showed that the models were not preserving enough information to distinguish between the digits. We rejected these models because they produced blurry and similar-looking images rather than clear reconstructions of each original digit.

### Overall Results
For the initial set of hyperparameter, we tested the differences that occured from changing the hidden layer's activation function, the final layer's activation function, and the loss function. These were tested with ***Adadelta*** as the optimizer. Now that we've narrowed down our model selection to 3 sets of activation functions that resulted in viable image reconstructions, we will be testing these with different optimizers to determine the best possible autoencoder.

In [ ]:
activation_function = ["relu"]
activation_final = ["sigmoid","softmax"]
loss_functions = ["binary_crossentropy","mean_squared_error"]
optimizer = ["adam","adagrad"]
autoencoders2 = dict()
i = 0
for act in activation_function:
    for act_fnl in activation_final:
        for loss in loss_functions:
            for opti in optimizer:
                if act_fnl == "softmax" and loss != "binary_crossentropy":
                    continue
                if i != 0:
                    w = autoencoders2[0]["decoded"].get_weights()
                else:
                    w = None
                autoencoders2[i] = create_autoencoder(img_data,act,act_fnl,loss,opti,w)
                i += 1
df_encoded_pars2 = pd.DataFrame(autoencoders2).transpose().drop(columns=["decoded","encoded"])

In [ ]:
plot_images(img_data,autoencoders2)
df_encoded_pars2

In [ ]:
activation_function = ["relu"]
activation_final = ["sigmoid","softmax"]
loss_functions = ["binary_crossentropy","mean_squared_error"]
optimizer = ["adam","adadelta"]
epochs = [50,100,150]
autoencoders3 = dict()
i = 0
for act in activation_function:
    for act_fnl in activation_final:
        for loss in loss_functions:
            for opti in optimizer:
                for epoch in epochs:
                    if (act_fnl == "softmax" and loss != "binary_crossentropy" and opti !="binary_crossentropy"):
                        continue
                    elif (act_fnl == "sigmoid" and opti == "adadelta" and loss != "binary_crossentropy"):
                        continue
                    if i != 0:
                        w = autoencoders3[0]["decoded"].get_weights()
                    else:
                        w = None
                    autoencoders3[i] = create_autoencoder(img_data,act,act_fnl,loss,opti,w,epoch)
                    i += 1
df_encoded_pars3 = pd.DataFrame(autoencoders3).transpose().drop(columns=["decoded","encoded"])

### Final Selection

## Conclusion

### Image Visualizations

## Final Propositions


## Acknowledgement of Contributions
### Group 3 Members:  
Sebastian Levy  
Idongesit Orok  
Miracle Okojie  

### Task Distribution:  

Load and Check Data - Sebastian Levy and Miracle Okojie
Normalize Data and Visualizations - Sebastian Levy  
Prepare Data for Encoding - Sebastian Levy  
Data Observations and Analysis - Sebastian Levy, Idongesit Orok, Miracle Okojie  
Encoder Architechture - Sebastian Levy  
Encoder Iteration - Sebastian Levy  
Encoder Iteration Analysys - Sebastian Levy, Idongesit Orok, Miracle Okojie  

We, Group 3, acknowledge that we understand all parts of this assignment, including those which we did not personally carry out.